# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

The Lane 2 contract (Refresh / Content Opportunity Scoring), in plain words:

1. **One row = one page** (`content_hash_id`) at one decision date. The underlying release records **page-days** (`report_date x client_hash_id x content_hash_id`); my lane's slice aggregates those page-days into one row per page, because the reviewer's decision happens per page.
2. **Tables I use:** `fact_content_daily_performance` (primary: one file per month partition) and `dim_clients` (context: per-client history starts). I never use `fact_content_daily_performance_sample` — it is exactly the final month (June 2026), the natural outcome window of any past->future label, so testing label logic there would mean developing inside my own test window.
3. **Time window (Week-3 mechanics slice, one mid-panel month):**

```
2026-03-01 ──── feature window (20 days) ──── 2026-03-20 | 2026-03-21 ──── target window (11 days) ──── 2026-03-31
                                                          ^
                                                 decision moment:
                                    nothing later than this day may enter a feature
```

   In the capstone this same design becomes prior-90-days -> next-30-days; the mechanics below are identical, only the window lengths change.
4. **What I'd rank:** a per-page priority score for the weekly review queue. The evaluation label carried next to it: **decline_11d** — TRUE when a page's mean daily GSC impressions over the target window (03-21..31) fall below 70% of its mean daily impressions over 03-14..20 (its own recent pace), with a minimum-volume floor of 100 impressions in the feature window. The outcome is measured in a window the features never touch; the thresholds are my written policy.
5. **One deliberate exclusion (and why):** the GA4 engagement columns, for now — page-days before a client's `ga4_data_start` carry search metrics only, with engagement zero-filled (`ga4_data_available = FALSE`); reading those zeros as "no engagement" would bake client onboarding into "page quality". (`fact_content_query_90d` is also deferred this week: its fixed 90-day window overlaps my target window.)

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Library pinning, token resolution (never typed in a cell), and local access to the release.
%pip -q install duckdb huggingface_hub
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from pathlib import Path
import subprocess

import duckdb

print("duckdb", duckdb.__version__)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found. One-time setup: (1) accept the gate at "
        "huggingface.co/datasets/FlyRank/internship-warehouse (affiliation 'FlyRank ML Internship 2026'); "
        "(2) create a READ token (Settings -> Access Tokens, type Read); "
        "(3) make it available: locally run  setx HF_TOKEN \"hf_...\"  in a fresh terminal, "
        "or in Colab add a Secret named HF_TOKEN. Then re-run this notebook."
    )
print("token found: present in environment (value not printed)")

import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm warns when jupyter widgets are absent

from huggingface_hub import snapshot_download

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    # Cloud-runtime fallback: the Colab badge opens just the notebook, not the repo.
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            print(f"repo not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tanay1122/flyrank-ml.git", str(target)], check=False, capture_output=True)
        if (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return target
    raise FileNotFoundError("No flyrank-ml repo found — run from inside the repo (or let the Colab fallback clone it).")

REPO = find_repo_root()
CACHE = REPO / "work" / "cache" / "warehouse"
LOCAL = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=["dim_clients.parquet", "fact_content_daily_performance/month=2026-03/*.parquet"],
    token=HF_TOKEN,
    local_dir=str(CACHE),
)
CACHE_DIR = Path(LOCAL)

DIM_CLIENTS = (CACHE_DIR / "dim_clients.parquet").as_posix()
FACT_MAR = (CACHE_DIR / "fact_content_daily_performance" / "month=2026-03" / "data_0.parquet").as_posix()
print("local files ready:")
print("  ", Path(FACT_MAR).name, "-", round(Path(FACT_MAR).stat().st_size / 1e6, 1), "MB (March partition)")
print("  ", Path(DIM_CLIENTS).name)

con = duckdb.connect()
dim_check = con.execute(f"SELECT COUNT(*) AS clients FROM read_parquet('{DIM_CLIENTS}')").df()
print(dim_check.to_string(index=False))
print("Expectation: 104 clients (the release's published dim_clients count) — the token works and the file is readable.")


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
duckdb 2.0.0
token found: present in environment (value not printed)


local files ready:
   data_0.parquet - 124.2 MB (March partition)
   dim_clients.parquet


 clients
     104
Expectation: 104 clients (the release's published dim_clients count) — the token works and the file is readable.


## 2. Fields: feature / label / context / excluded

Every field this week touches, one bucket each (per the data-contract skill):

| Bucket | Field(s) | Read as |
|---|---|---|
| **Feature** (GSC page-days, decision moment 2026-03-20) | `gsc_impressions` (summed per window) | demand known so far |
| **Feature** | `gsc_clicks` (summed) | click capture known so far |
| **Feature** | `gsc_avg_position` (mean over non-null row-days) | SERP standing so far |
| **Feature** | row counts per date bucket (presence + recency pace) | regularity/pace known so far |
| **Label** | `decline_11d` — computed from target-window impressions only | the thing I rank against; never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | grouping, joining, windows — never learnable |
| **Context** | `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | row-level availability flags: read for filtering, never features |
| **Excluded** | GA4 engagement columns | zero-filled before `ga4_data_start`; excluded per contract answer 5 |
| **Excluded** | `fact_content_daily_performance_sample` | the sealed final month; no label logic there, ever |
| **Excluded** | `fact_content_query_90d` (this week) | its fixed 90-day window overlaps my target window |
| **Excluded** | product decision columns (`health_score`, flags) | not shipped in the release; if ever rebuilt, baseline-to-beat only, never a feature |

**Missing values (checked with queries below, not assumed):** a page's absent days are *absent rows*, not zeros — but the release writes rows it has no data for anyway, and zero-fills them: query 3 shows that every row-day with `gsc_data_available` not TRUE carries `gsc_impressions = 0`, with `gsc_avg_position` NULL on exactly those rows. Any aggregate that ignores the flag reads fake zeros as market behavior — so every feature/target query below builds on `gsc_data_available IS TRUE` rows only. The code cell does the schema introspection (an identity check, not one of the three verification queries).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Schema introspection of the March partition: which columns exist, are they the ones the contract names?
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_MAR}')").df()
print(f"March partition schema: {len(schema)} columns")
print(", ".join(schema.column_name.tolist()))

expected = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
            "gsc_avg_position", "gsc_data_available", "ga4_data_available", "client_has_gsc", "client_has_ga4"]
missing = [c for c in expected if c not in set(schema.column_name)]
print("Contract fields all present:", not missing, ("| missing: " + ", ".join(missing)) if missing else "")
ga4_cols = [c for c in schema.column_name if c.startswith("ga4_")][:10]
print("GA4 columns (excluded this week, listed for the record):", ga4_cols)

March partition schema: 31 columns
report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month
Contract fields all present: True 
GA4 columns (excluded this week, listed for the record): ['ga4_data_available', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on the mid-panel month `month=2026-03`, exactly as the card orders — grain, row count + date span, availability (with `IS TRUE`). Each cell prints its output and the expectation it is checked against.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Verification query 1 of 3 — GRAIN. Zero duplicate (date, client, content) keys => one row really is one page-day.
dup_keys = con.execute(f'''
    SELECT COUNT(*) AS duplicate_keys
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM read_parquet('{FACT_MAR}')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
        LIMIT 5
    )
''').df()
print(dup_keys.to_string(index=False))
print("Expectation: duplicate_keys = 0 -> the fact table's grain (page-day) holds, so my per-page aggregates count each day exactly once.")

 duplicate_keys
              0
Expectation: duplicate_keys = 0 -> the fact table's grain (page-day) holds, so my per-page aggregates count each day exactly once.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Verification query 2 of 3 — ROW COUNT + DATE SPAN for my slice, March 2026.
stats = con.execute(f'''
    SELECT COUNT(*) AS rows_month,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM read_parquet('{FACT_MAR}')
''').df()
print(stats.to_string(index=False))
print("Expectations: span = 2026-03-01..2026-03-31 exactly; clients <= 104 (the release has 104); "
      "pages <= 519,606 (the release's deduplicated count); rows ~ a mid-panel month's share of ~79M rows.")

 rows_month  first_day   last_day  clients  pages
    9841378 2026-03-01 2026-03-31       55 331437
Expectations: span = 2026-03-01..2026-03-31 exactly; clients <= 104 (the release has 104); pages <= 519,606 (the release's deduplicated count); rows ~ a mid-panel month's share of ~79M rows.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Verification query 3 of 3 — AVAILABILITY, null-safe with IS TRUE: what exists, and what those rows contain.
avail = con.execute(f'''
    SELECT COUNT(*) AS rows_month,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_true,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true,
           SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_flag_null_unknown,
           SUM(CASE WHEN (gsc_data_available IS TRUE) AND (ga4_data_available IS TRUE) THEN 1 ELSE 0 END) AS both_available_true,
           SUM(CASE WHEN (gsc_data_available IS TRUE) AND (ga4_data_available IS TRUE)
                          AND (gsc_impressions > 0) THEN 1 ELSE 0 END) AS lane_slice_row_days,
           SUM(CASE WHEN NOT (gsc_data_available IS TRUE) AND (gsc_impressions = 0) THEN 1 ELSE 0 END) AS fake_zero_row_days,
           SUM(CASE WHEN (gsc_data_available IS TRUE) AND (gsc_impressions = 0) THEN 1 ELSE 0 END) AS real_zero_imp_true_days
    FROM read_parquet('{FACT_MAR}')
''').df()
total = avail["rows_month"][0]
for col in avail.columns[1:]:
    value = avail[col][0]
    print(f"{col:>26}: {int(value):>12,} row-days ({value / total * 100:.1f}%)")
print("Expectations: `IS TRUE` is null-safe, so a NULL flag can never count as available (ga4_flag_null_unknown"); 
print("shows how much status is simply unknown). fake_zero_row_days should equal every row the release cannot")
print("see — those rows are ZERO-FILLED, not absent, which is why aggregates must filter `IS TRUE` first.")
print("real_zero_imp_true_days = 0 means a flag-TRUE day always carries real impressions this month;")
print("lane_slice_row_days is the only raw material engagement features could ever legitimately touch.")

        gsc_available_true:    3,611,061 row-days (36.7%)
        ga4_available_true:      413,966 row-days (4.2%)
     ga4_flag_null_unknown:    3,018,741 row-days (30.7%)
       both_available_true:      364,347 row-days (3.7%)
       lane_slice_row_days:      364,347 row-days (3.7%)
        fake_zero_row_days:    6,230,317 row-days (63.3%)
   real_zero_imp_true_days:            0 row-days (0.0%)
Expectations: `IS TRUE` is null-safe, so a NULL flag can never count as available (ga4_flag_null_unknown
shows how much status is simply unknown). fake_zero_row_days should equal every row the release cannot
see — those rows are ZERO-FILLED, not absent, which is why aggregates must filter `IS TRUE` first.
real_zero_imp_true_days = 0 means a flag-TRUE day always carries real impressions this month;
lane_slice_row_days is the only raw material engagement features could ever legitimately touch.


## Three verification queries done. Now the five features, each with its availability line (the card's rule)

The leak-free, decision-moment-safe feature frame aggregates March 1-20 row-days that pass `gsc_data_available IS TRUE`
(query 3's zero-filled rows never enter the windows), then applies a written aggregation floor:
`SUM(impressions) >= 100` AND `COUNT(*) >= 10` — at least 10 of the 20 feature-window days really present —
a policy I wrote down so tiny pages cannot vote. Five features, max:

| # | Feature | Formula (over 03-01..20 only) | Knowable at the decision moment because |
|---|---|---|---|
| 1 | `imp_early_tot` | SUM of daily `gsc_impressions` | the query window closes on 2026-03-20; no row after the decision date is read |
| 2 | `click_rate_pct` | 100 x SUM(clicks)/SUM(impressions) | both sums come from the feature window only |
| 3 | `pos_mean` | AVG(`gsc_avg_position`) (nulls skipped, never zero-filled) | averages only pre-decision row-days |
| 4 | `days_active` | COUNT(*) of kept row-days (each is a day the page really showed up) | counts history that already happened |
| 5 | `momentum` | mean daily impressions 03-14..20 divided by mean daily impressions 03-01..13 | the latest day it reads is 03-20 — the decision date itself |

One footgun declared up front: `momentum` shares its 03-14..20 part with the label's baseline window. That is legal (it is inside the feature window and never overlaps the target), but it is exactly the kind of grey zone I re-audit in the leakage notebook — if the honest score later collapses when `momentum` leaves, that means the feature was carrying the label, not information.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Build the per-page feature frame (small result, D beats RAM: SQL aggregates page-days, pandas gets pages only).
import sys

features = con.execute(f'''
    WITH early AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '2026-03-20' AND gsc_data_available IS TRUE
    ), aggregated AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_early_tot,
               SUM(gsc_clicks) AS clicks_early_tot,
               AVG(CASE WHEN report_date >= DATE '2026-03-14' THEN gsc_impressions END) AS pace_mean,
               AVG(CASE WHEN report_date <= DATE '2026-03-13' THEN gsc_impressions END) AS base_mean,
               AVG(gsc_avg_position) AS pos_mean,
               SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_active
        FROM early
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND COUNT(*) >= 10
    )
    SELECT *,
           pace_mean / NULLIF(base_mean, 0) AS momentum,
           100.0 * clicks_early_tot / imp_early_tot AS click_rate_pct
    FROM aggregated
''').df()

feature_cols = ["imp_early_tot", "click_rate_pct", "pos_mean", "days_active", "momentum"]
print(f"Feature frame: {len(features):,} pages x {len(feature_cols)} features")
one_row_one_page = features["content_hash_id"].is_unique
print(f"Built-frame grain check: content_hash_id unique -> {one_row_one_page} (one row really is one page)")
preview = features[feature_cols].head(5).copy()
preview[["click_rate_pct", "pos_mean", "momentum"]] = preview[["click_rate_pct", "pos_mean", "momentum"]].round(3)
print(preview.to_string(index=False))

Feature frame: 83,609 pages x 5 features
Built-frame grain check: content_hash_id unique -> True (one row really is one page)
 imp_early_tot  click_rate_pct  pos_mean  days_active  momentum
         202.0           2.475     3.664         20.0     0.430
         470.0           0.000    59.276         18.0     0.284
         383.0           0.000    70.777         18.0     1.023
        2122.0           0.000    58.397         18.0     0.794
        8401.0           1.071     3.589         18.0     1.014


## Now the trap — one label-derived column, added on purpose

`decline_11d` is TRUE when the page's mean daily impressions over the target window (03-21..31) fall below 70% of its 03-14..20 pace. The engine of that rule is just a ratio between the two windows. So the "feature" for the trap is:

- **`trap_target_ratio` = mean daily impressions 03-14..20 / mean daily impressions 03-21..31** — the label's own arithmetic wearing a feature costume, flipped so a HIGH value means decline risk (the label is literally TRUE when this exceeds ~1.43). Its numerator is knowable at the decision moment; its denominator does not exist until the target window closes. Watch what the quick score does.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Compute the label's target window, add the trap column, watch the leap, delete it, keep the honest number.
import pandas as pd

target = con.execute(f'''
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_target_tot,
           AVG(gsc_impressions) AS target_mean
    FROM read_parquet('{FACT_MAR}')
    WHERE report_date >= DATE '2026-03-21' AND report_date <= DATE '2026-03-31'
          AND gsc_data_available IS TRUE
    GROUP BY 1, 2
''').df()

frame = features.merge(target, on=["client_hash_id", "content_hash_id"], how="left")
no_outcome = int(frame["target_mean"].isna().sum())
print(f"Pages from the feature frame with NO rows in the target window (absent, not zero): {no_outcome:,} — dropped from evaluation")
frame = frame[frame["target_mean"].notna()].copy()
outcome_ok = frame["pace_mean"] > 0
print(f"Pages whose 03-14..20 pace is zero or undefined (NULL): {int((~outcome_ok).sum()):,} — the decline ratio would be unanswerable, dropped too")
frame = frame[outcome_ok].copy()
frame["trap_target_ratio"] = frame["pace_mean"] / frame["target_mean"]
frame["decline_11d"] = (frame["target_mean"] < 0.70 * frame["pace_mean"]).astype(int)

n_pos = int(frame["decline_11d"].sum())
print(f"Evaluation frame: {len(frame):,} pages; positives = {n_pos:,} ({n_pos / len(frame) * 100:.1f}% base rate)")

def auc(truth: pd.Series, scores: pd.Series) -> float:
    ranks = scores.rank()
    n_pos = int((truth == 1).sum())
    n_neg = int((truth == 0).sum())
    return float((ranks[truth == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import percentile_rank

scaled = pd.DataFrame({col: percentile_rank(frame[col]) for col in feature_cols})

auc_honest = auc(frame["decline_11d"], scaled[feature_cols].mean(axis=1))
scaled_trap = scaled.copy()
scaled_trap["trap_target_ratio"] = percentile_rank(frame["trap_target_ratio"])
auc_plus_trap = auc(frame["decline_11d"], scaled_trap.mean(axis=1))
auc_trap_alone = auc(frame["decline_11d"], frame["trap_target_ratio"])

print(f"Quick score (AUC of the rank-averaged 5-feature score): {auc_honest:.3f}")
print(f"Quick score with the trap column added:                {auc_plus_trap:.3f}  <- the jump, by construction")
print(f"Trap column alone:                                     {auc_trap_alone:.3f}  <- it IS the label's own rule")
for col in feature_cols:
    print(f"  single-feature AUC {col}: {auc(frame['decline_11d'], frame[col].fillna(0).rank()):.3f}")
del scaled_trap["trap_target_ratio"]
print("Trap column DELETED. The kept number is the honest one:")
print(f"  honest AUC = {auc_honest:.3f} — real, owned, argued about for weeks (coin-flip ordering scores 0.500;")
print(f"  {int(frame['decline_11d'].sum()):,} of {len(frame):,} kept pages decline ({frame['decline_11d'].mean() * 100:.1f}%) under the written 70%-pace rule).")

Pages from the feature frame with NO rows in the target window (absent, not zero): 314 — dropped from evaluation
Pages whose 03-14..20 pace is zero or undefined (NULL): 37 — the decline ratio would be unanswerable, dropped too
Evaluation frame: 83,258 pages; positives = 13,277 (15.9% base rate)
Quick score (AUC of the rank-averaged 5-feature score): 0.442
Quick score with the trap column added:                0.665  <- the jump, by construction
Trap column alone:                                     1.000  <- it IS the label's own rule
  single-feature AUC imp_early_tot: 0.392


  single-feature AUC click_rate_pct: 0.407
  single-feature AUC pos_mean: 0.520
  single-feature AUC days_active: 0.477
  single-feature AUC momentum: 0.585
Trap column DELETED. The kept number is the honest one:
  honest AUC = 0.442 — real, owned, argued about for weeks (coin-flip ordering scores 0.500;
  13,277 of 83,258 kept pages decline (15.9%) under the written 70%-pace rule).


## 4. Data limits

**One named limitation, quantified: the release is an unbalanced panel — different clients' histories start on different dates.** A global calendar window quietly assumes every page has a real prior window; for a client who started tracking after 2025-09-30, the "prior 90 days" of the capstone design does not exist yet, and demand/pace features computed anyway would describe *coverage*, not the page. The query below counts exactly those clients and their weight in my March slice. (The same thinness repeats inside every month: query 3 showed the engagement side of the river carries far fewer honest row-days than the search side in a mid-panel month.)

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Quantify the unbalanced-panel limitation: short-history clients and their March weight.
releases = con.execute(f'''
SELECT client_hash_id, gsc_data_start
FROM read_parquet('{DIM_CLIENTS}')
''').df()
late = releases[releases["gsc_data_start"] > pd.Timestamp("2025-09-30")]
no_start = int(releases["gsc_data_start"].isna().sum())
print(f"dim_clients: {len(releases)} clients; {len(late)} start GSC tracking after 2025-09-30 "
      f"(< 6 months of history before the March decision point); {no_start} have NO gsc_data_start at all")

weights = con.execute(f'''
    SELECT client_hash_id, COUNT(*) AS rows_mar
    FROM read_parquet('{FACT_MAR}')
    GROUP BY 1
''').df()
late_share = weights[weights["client_hash_id"].isin(late["client_hash_id"])]["rows_mar"].sum() / weights["rows_mar"].sum()
print(f"Their weight in the March partition: {late_share * 100:.1f}% of row-days")
print("Contract consequence: capstone windows must be per-client (anchored on gsc_data_start), never one global calendar window.")

dim_clients: 104 clients; 43 start GSC tracking after 2025-09-30 (< 6 months of history before the March decision point); 37 have NO gsc_data_start at all
Their weight in the March partition: 38.1% of row-days
Contract consequence: capstone windows must be per-client (anchored on gsc_data_start), never one global calendar window.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.